# Run Trained Model Demo

This notebook demonstrates how to load the trained U-Net model and run inference on a sample image.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
from model_utils import UNet

# Configuration
MODEL_PATH = "../models/best_model.pth"
INPUT_IMAGE_PATH = "sample_plot.png"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {DEVICE}")

## 1. Load and Preprocess Image

In [ ]:
def load_and_preprocess_image(image_path):
    img = Image.open(image_path).convert('RGB')
    
    # Handle valid sample_plot.png composite by cropping
    if img.size[0] > 1500 and "sample_plot" in str(image_path):
        print("Detected composite plot. Cropping first panel...")
        width = img.size[0] // 3
        img = img.crop((0, 0, width, img.size[1]))
    
    # Resize to 512x512
    img = img.resize((512, 512), Image.Resampling.BILINEAR)
    
    # Normalize
    img_np = np.array(img).astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img_np = (img_np - mean) / std
    
    # To Tensor [1, C, H, W]
    img_tensor = torch.from_numpy(img_np.transpose(2, 0, 1)).unsqueeze(0)
    return img_tensor, np.array(img)

input_tensor, display_img = load_and_preprocess_image(INPUT_IMAGE_PATH)
plt.imshow(display_img)
plt.title("Input Image")
plt.axis('off')
plt.show()

## 2. Load Model

In [ ]:
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
num_classes = checkpoint['model_state_dict']['final.weight'].shape[0] if 'model_state_dict' in checkpoint else 4

model = UNet(num_classes=num_classes)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(DEVICE)
model.eval()
print("Model loaded.")

## 3. Run Inference

In [ ]:
with torch.no_grad():
    logits = model(input_tensor.to(DEVICE))
    pred_mask = torch.argmax(logits, dim=1).squeeze().cpu().numpy()

# Color map
colors = np.array([[0,0,0], [0,0,255], [0,255,0], [255,0,0]], dtype=np.uint8)
h, w = pred_mask.shape
colored_mask = np.zeros((h, w, 3), dtype=np.uint8)
for c in range(min(num_classes, 4)):
    colored_mask[pred_mask == c] = colors[c]

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(display_img)
plt.title("Input")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(colored_mask)
plt.title("Prediction")
plt.axis('off')
plt.show()